## CREATE DATABASE SQLite (Directly)

In [5]:
import sqlite3
import pandas as pd

# ============================================================
# 1. CONNECT TO DATABASE
# ============================================================
conn = sqlite3.connect("olist.db")
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = OFF;")


# ============================================================
# 2. DROP EXISTING TABLES
# ============================================================
tables = [
    "customers",
    "geolocation",
    "sellers",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products"
]

for t in tables:
    cursor.execute(f"DROP TABLE IF EXISTS {t};")
print("All tables dropped.")


# ============================================================
# 3. CREATE TABLES (Olist schema)
# ============================================================

cursor.execute("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT,
    customer_zip_code_prefix TEXT,
    customer_city TEXT,
    customer_state TEXT
);
""")

cursor.execute("""
CREATE TABLE geolocation (
    geolocation_zip_code_prefix TEXT,
    geolocation_lat REAL,
    geolocation_lng REAL,
    geolocation_city TEXT,
    geolocation_state TEXT
);
""")

cursor.execute("""
CREATE TABLE sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix TEXT,
    seller_city TEXT,
    seller_state TEXT
);
""")

cursor.execute("""
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_status TEXT,
    order_purchase_timestamp TEXT,
    order_approved_at TEXT,
    order_delivered_carrier_date TEXT,
    order_delivered_customer_date TEXT,
    order_estimated_delivery_date TEXT,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);
""")

cursor.execute("""
CREATE TABLE order_items (
    order_id TEXT,
    order_item_id INTEGER,
    product_id TEXT,
    seller_id TEXT,
    shipping_limit_date TEXT,
    price REAL,
    freight_value REAL,
    PRIMARY KEY(order_id, order_item_id),
    FOREIGN KEY(order_id) REFERENCES orders(order_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id),
    FOREIGN KEY(seller_id) REFERENCES sellers(seller_id)
);
""")

cursor.execute("""
CREATE TABLE order_payments (
    order_id TEXT,
    payment_sequential INTEGER,
    payment_type TEXT,
    payment_installments INTEGER,
    payment_value REAL,
    PRIMARY KEY(order_id, payment_sequential),
    FOREIGN KEY(order_id) REFERENCES orders(order_id)
);
""")

cursor.execute("""
CREATE TABLE order_reviews (
    review_id TEXT PRIMARY KEY,
    order_id TEXT,
    review_score INTEGER,
    review_creation_date TEXT,
    review_answer_timestamp TEXT,
    FOREIGN KEY(order_id) REFERENCES orders(order_id)
);
""")

cursor.execute("""
CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_description_lenght REAL,
    product_weight_g REAL,
    product_length_cm REAL,
    product_height_cm REAL,
    product_width_cm REAL,
    product_category_name_english TEXT
);
""")

print("All tables created.")


# ============================================================
# 4. DUPLICATE CLEANING RULES
# ============================================================
PK_MAP = {
    "customers": "customer_id",
    "geolocation": None,
    "sellers": "seller_id",
    "orders": "order_id",
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": "review_id",
    "products": "product_id"
}

def clean_duplicates(df, table):
    pk = PK_MAP[table]

    if pk is None:
        return df.drop_duplicates()

    if isinstance(pk, list):  # composite key
        return df.drop_duplicates(subset=pk)

    return df.drop_duplicates(subset=[pk])


# ============================================================
# 5. IMPORT CSV WITH AUTO-COLUMN-MATCHING
# ============================================================
def import_csv(csv_path, table):
    print(f"\nProcessing {table} ...")

    df = pd.read_csv(csv_path)

    # --- 1. Read SQLite table column names
    cursor.execute(f"PRAGMA table_info({table});")
    sqlite_cols = [col[1] for col in cursor.fetchall()]

    # --- 2. Keep only matching columns (prevents errors)
    df = df[[c for c in df.columns if c in sqlite_cols]]

    # --- 3. Clean duplicates
    df = clean_duplicates(df, table)

    # --- 4. Insert into SQLite
    df.to_sql(table, conn, if_exists='append', index=False)

    print(f"Imported {len(df)} rows into {table}.")


# ============================================================
# 6. RUN IMPORT FOR ALL FILES
# ============================================================
import_csv("clean_customers.csv", "customers")
import_csv("clean_geolocation.csv", "geolocation")
import_csv("clean_sellers.csv", "sellers")
import_csv("clean_orders.csv", "orders")
import_csv("clean_order_items.csv", "order_items")
import_csv("clean_order_payments.csv", "order_payments")
import_csv("clean_order_review.csv", "order_reviews")
import_csv("clean_products.csv", "products")


# ============================================================
# 7. RE-ENABLE FOREIGN KEYS
# ============================================================
cursor.execute("PRAGMA foreign_keys = ON;")
conn.commit()
conn.close()

print("\n🎉 SQLite database built successfully!")

All tables dropped.
All tables created.

Processing customers ...
Imported 99441 rows into customers.

Processing geolocation ...
Imported 738332 rows into geolocation.

Processing sellers ...
Imported 3095 rows into sellers.

Processing orders ...
Imported 99441 rows into orders.

Processing order_items ...
Imported 112650 rows into order_items.

Processing order_payments ...
Imported 103883 rows into order_payments.

Processing order_reviews ...
Imported 98410 rows into order_reviews.

Processing products ...
Imported 32951 rows into products.

🎉 SQLite database built successfully!
